# Audio Restoration for MusicGen Outputs

**Machine Learning 2024/2025 — Sapienza University of Rome**  
Project ID 2: Audio Restoration for Generative Models

This notebook is a short, self-contained walkthrough of the current zero-shot
pipeline. It synthesizes a small musical clip, degrades it the way generative or
low-bitrate audio often fails (hiss, quantization, band limitation, clipping),
restores it, and reports objective metrics with the same code used by the CLI.


## 1. Setup

If you opened this notebook in Colab, clone the repository and install the core
dependencies first. When running locally from the repository root, this cell only
adds the root to `sys.path`.


In [ ]:
# Colab only (uncomment):
# !git clone https://github.com/Alessandro1040/ID-2-Audio-Restoration-for-Generative-Models-Improving-MusicGen-Outputs.git
# %cd ID-2-Audio-Restoration-for-Generative-Models-Improving-MusicGen-Outputs
# !pip install -q -r requirements.txt

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repository:", ROOT)


## 2. Generate a reproducible degraded clip

`src.demo.synth_music` creates a chord progression with bass and percussion.
`src.demo.corrupt` applies the failure modes we want to test: an 8 kHz band
limit, broadband hiss at 12 dB SNR, 8-bit quantization and clipping.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf

from src.demo import synth_music, corrupt
from src import AudioRestorer, AudioQualityMetrics

SR = 32000
clean = synth_music(SR, duration=12.0, seed=7)
dirty = corrupt(clean, sr=SR, noise_db=12.0, bits=8, drive=2.4, band=8000.0)
print("clean", clean.shape, "dirty", dirty.shape, "duration", len(clean) / SR, "s")


## 3. Run the restoration

The default `music` preset preserves the sample rate and channel layout. The
returned report is JSON-serializable and contains per-stage settings, clipping
statistics and loudness measurements.


In [ ]:
restorer = AudioRestorer(preset="music", verbose=True)
restored, report = restorer.restore_array(dirty, SR)
print("sample rate:", report["sample_rate"], "channels:", report["channels"])
print("declip:", report["stages"].get("declip"))
print("loudness:", report["loudness"])


## 4. Objective comparison

The clean signal is known here, so we can compute reference-based metrics:
SI-SDR, SNR, segmental SNR and log-spectral distance (LSD). The table also
contains reference-free descriptors; their direction is declared in
`src.metrics.METRIC_DIRECTION`, so a lower-is-better change is never shown as an
improvement by accident.


In [ ]:
metrics = AudioQualityMetrics(sample_rate=SR)
original_metrics, restored_metrics, improvements = metrics.compare_arrays(
    clean, dirty, restored)
metrics.print_comparison(original_metrics, restored_metrics, improvements)


## 5. Listen and look

The spectrogram is the most direct diagnostic: broadband hiss appears as a
uniform haze, clipping adds vertical stripes, and the restoration should reduce
the haze without erasing the harmonic lines.


In [ ]:
from src.dsp import as_mono

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, sharey=True)
for ax, data, title in zip(
        axes, (clean, dirty, restored),
        ("Clean reference", "Degraded input", "Restored")):
    data = as_mono(data)  # restore_array returns (n, 1)
    spectrum = librosa.amplitude_to_db(
        np.abs(librosa.stft(data, n_fft=2048, hop_length=512)), ref=np.max)
    image = librosa.display.specshow(
        spectrum, sr=SR, hop_length=512, x_axis="time", y_axis="log",
        ax=ax, cmap="magma", vmin=-80, vmax=0)
    ax.set_title(title)
fig.colorbar(image, ax=axes, format="%+2.0f dB", shrink=0.8)
plt.show()


In [ ]:
try:
    from IPython.display import Audio, display
except ImportError:
    Audio = None

if Audio is None:
    print("Install IPython/Jupyter to play the audio inline.")
else:
    print("Degraded input")
    display(Audio(dirty, rate=SR))
    print("Restored")
    display(Audio(restored[:, 0], rate=SR))


## 6. Use it on real files

For a single file, the file API writes a 24-bit WAV and a report:

```python
report = restorer.restore_file("input.wav", "restored.wav")
```

From a shell, the same pipeline is available as:

```bash
python scripts/restore_audio.py input.wav -o restored/
python scripts/restore_audio.py generated/ -o restored/ --sample-rate 32000 --mono --target-lufs -16
python scripts/evaluate.py --original_dir dirty/ --restored_dir restored/ --reference_dir clean/
```

Always listen to the result and inspect the reference-free descriptors. The
pipeline is conservative by design; for very noisy material compare the
`gentle`, `music` and `strong` presets instead of assuming that stronger
denoising is better.
